# ACCESS-MOPPy Getting Started

Welcome to the ACCESS-MOPPy Getting Started guide!

This notebook will walk you through the initial setup and basic usage of ACCESS-MOPPy, a tool designed to post-process ACCESS model output and produce CMIP-compliant datasets. You’ll learn how to configure your environment, prepare your data, and run the CMORisation workflow using both the Python API and Dask for scalable processing.

By following this guide, you’ll be able to:
- Set up your user configuration
- Select input data files
- Run the CMORisation process for selected variables
- Inspect and save the processed output


## Set up configuration

When you first import `access_moppy` in a Python environment, the package will automatically create a `user.yml` file in your home directory (`~/.moppy/user.yml`).  

During this initial setup, you will be prompted to provide some basic information, including:  
- Your name  
- Your email address  
- Your work organization
- Your ORCID

This information is stored in `user.yml` and will be used as global attributes in the files generated during the CMORisation process. This ensures that each CMORised file includes metadata identifying who performed the CMORisation, allowing us to track data provenance and follow up with the responsible person if needed.

In [1]:
from access_moppy import ACCESS_ESM_CMORiser


Loaded Configuration:
Creator Name: Romain Beucher
Organisation: ACCESS-NRI
Creator Email: romain.beucher@anu.edu.au
Creator URL: www.google.com


## Dask support

ACCESS-MOPPeR supports Dask for parallel processing, which can significantly speed up the CMORisation workflow, especially when working with large datasets. To use Dask with ACCESS-MOPPeR, you can create a Dask client it will be used to manage the distributed computation. This allows you to take advantage of multiple CPU cores or even a cluster of machines, depending on your setup.
You can configure the Dask client to use a specific number of threads per worker, which can help optimize performance based on your hardware and the size of the datasets you are processing.

Here's an example of how to set up a Dask client:

```python
import dask.distributed as dask

client = dask.Client(threads_per_worker=1)
client
```

In [2]:
import dask.distributed as dask

client = dask.Client(threads_per_worker=1)
client

/home/romain/PROJECTS/ACCESS-MOPPeR/mopper_conda/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46051 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:46051/status,
Dashboard: http://127.0.0.1:46051/status,Workers: 12
Total threads: 12,Total memory: 15.06 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42637,Workers: 12
Dashboard: http://127.0.0.1:46051/status,Total threads: 12
Started: Just now,Total memory: 15.06 GiB
Comm: tcp://127.0.0.1:43365,Total threads: 1
Dashboard: http://127.0.0.1:38401/status,Memory: 1.25 GiB
Nanny: tcp://127.0.0.1:42329,


## Data selection

The `ACCESS_ESM_CMORiser` class (described in detail below) takes as input a list of paths to NetCDF files containing the raw model output variables to be CMORised. The CMORiser does **not** assume any specific folder structure, DRS (Data Reference Syntax), or file naming convention. It is intentionally left to the user to ensure that the provided files contain the original variables required for CMORisation.

For this example, we will be using data from the **December Spin-up run for ACCESS-ESM1.6 development**. This dataset is available on NCI at `/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control` and represents an important milestone in the ACCESS-ESM1.6 model development process. Using this dataset allows us to demonstrate CMORisation with realistic ACCESS model output that's being prepared for CMIP7.

This design is intentional: ACCESS-NRI plans to integrate ACCESS-MOPPeR into extended workflows that leverage the [ACCESS-NRI Intake Catalog](https://github.com/ACCESS-NRI/access-nri-intake-catalog) or evaluation frameworks such as [ESMValTool](https://www.esmvaltool.org/) and [ILAMB](https://www.ilamb.org/). By decoupling file selection from the CMORiser, ACCESS-MOPPeR can be flexibly used in a variety of data processing and evaluation pipelines.

In [ ]:
# Define the root folder path for the ACCESS-ESM1.6 December Spin-up dataset
# This path will be used consistently throughout the notebook for all model components
ROOT_FOLDER = "/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/"

# Define target folder patterns for different model components
ATMOS_FOLDERS = "output40[0-1]/atmosphere/netCDF/"
OCEAN_FOLDERS = "output40[0-9]/ocean/"
LAND_FOLDERS = "output40[0-1]/atmosphere/netCDF/"
OCEAN_GRID_FOLDERS = "output401/ocean/"  # Specific folder for ocean grid files

print(f"Using dataset from: {ROOT_FOLDER}")
print(f"Target folders:")
print(f"  Atmosphere: {ATMOS_FOLDERS}")
print(f"  Ocean: {OCEAN_FOLDERS}")
print(f"  Land: {LAND_FOLDERS}")
print(f"  Ocean Grid: {OCEAN_GRID_FOLDERS}")

In [ ]:
# Here we use NetCDF files from the ACCESS-ESM1.6 December Spin-up run
# This data is from the development of ACCESS-ESM1.6 for CMIP7
import glob

files = glob.glob(ROOT_FOLDER + ATMOS_FOLDERS + "aiihca.pa-*_mon.nc")
print(f"Found {len(files)} atmospheric files for CMORisation")

Found 0 files for CMORisation


### Parent experiment information

In CMIP workflows, providing parent experiment information is required for proper data provenance and traceability. This metadata describes the relationship between your experiment and its parent (for example, a historical run branching from a piControl simulation), and is essential for CMIP data publication and compliance.

However, for some applications—such as when using ACCESS-MOPPeR to interact with evaluation frameworks like [ESMValTool](https://www.esmvaltool.org/) or [ILAMB](https://www.ilamb.org/)—strict CMIP compliance is not always necessary. In these cases, you may choose to skip providing parent experiment information to simplify the workflow.

If you choose to skip this step, ACCESS-MOPPeR will issue a warning to let you know that, if you write the output to disk, the resulting file may not be compatible with CMIP requirements for publication. This flexibility allows you to use ACCESS-MOPPeR for rapid evaluation and prototyping, while still supporting full CMIP compliance when needed.

In [5]:
parent_experiment_config = {
    "parent_experiment_id": "piControl-spinup", # <-- This is for placeholder.
    "parent_activity_id": "CMIP",
    "parent_source_id": "ACCESS-ESM1-5", # <-- ACCESS-ESM1-6 is not yet registered in the CMIP6 catalog, so we use ACCESS-ESM1-5 as a placeholder
    "parent_variant_label": "r1i1p1f1",
    "parent_time_units": "days since 0001-01-01 00:00:00",
    "parent_mip_era": "CMIP6",
    "branch_time_in_child": 0.0, # <-- This is for placeholder. The actual branch time in child should be the time of the first time step in the input data, converted to the parent time units.
    "branch_time_in_parent": 54786.0, # <-- This is the branch time in parent for the December Spin-up run, which is the time of the first time step in the input data (1850-12-01), converted to the parent time units (days since 0001-01-01).
    "branch_method": "standard",
}

## Set up the CMORiser for CMORisation

To begin the CMORisation process, you need to create an instance of the `ACCESS_ESM_CMORiser` class. This class requires several key parameters, including the list of input NetCDF files and metadata describing your experiment.

A crucial parameter is the `compound_name`, which should be specified using the full CMIP convention: `table.variable` (for example, `Amon.rsds`). This format uniquely identifies the variable, its frequency (e.g., monthly, daily), and the associated CMIP table, ensuring that all requirements for grids and metadata are correctly handled. Using the full compound name helps avoid ambiguity and guarantees that the CMORiser applies the correct standards for each variable.

You can also provide additional metadata such as `experiment_id`, `source_id`, `variant_label`, and `grid_label` to ensure your output is CMIP-compliant. Optionally, you may include parent experiment information for full provenance tracking.

In [6]:
cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,
    compound_name="Amon.rsds",
    experiment_id="piControl-spinup",
    source_id="ACCESS-ESM1-5", # <-- ACCESS-ESM1-6 is not yet registered in the CMIP6 catalog, so we use ACCESS-ESM1-5 as a placeholder
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,  # <-- This is optional, can be skipped if not needed
)

## Exploring Variable Mappings

ACCESS-MOPPy provides an enhanced variable mapping display that helps you understand how your raw model variables are mapped to CMIP-compliant variables. The `variable_mapping` attribute now provides a rich, interactive display in Jupyter notebooks that shows:

- Variable metadata (CF standard names, units, dimensions)
- Mapping completeness and validation status
- Model-specific mapping information
- Easy-to-read tabular format with color coding

Let's explore the variable mapping for our selected variable:

In [7]:
# Display the variable mapping with enhanced formatting
# This shows the mapping between raw model variables and CMIP standards
cmoriser.variable_mapping

Model:,ACCESS-ESM1.6
Variable:,rsds
CF Standard Name:,surface_downwelling_shortwave_flux_in_air
Units:,W m-2
Dimensions:,"time: time, lat: lat, lon: lon"
Model Variable:,fld_s01i235
Processing:,DIRECT
Formula:,fld_s01i235
Positive:,down


The variable mapping display above shows:
- **Variable Name**: The CMIP variable name (rsds - surface downwelling shortwave flux)
- **CF Standard Name**: The Climate and Forecast conventions standard name
- **Units**: Expected units for the CMIP-compliant variable
- **Dimensions**: How the data dimensions map between raw and CMIP formats
- **Model Info**: Shows the ACCESS model version used for this mapping

### Exploring Different Variables

Let's see how the variable mapping display looks for different types of variables. Here's an example with atmospheric temperature (tas):

In [9]:
# Create a CMORiser for a different variable to see mapping differences
tas_cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,  # Can reuse the same files if they contain temperature data
    compound_name="Amon.tas",  # Near-surface air temperature
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
)

# Display the mapping for atmospheric temperature
tas_cmoriser.variable_mapping

/home/romain/PROJECTS/ACCESS-MOPPeR/ACCESS-MOPPy/src/access_moppy/driver.py:162: UserWarning: No parent_info provided. Defaulting to piControl parent experiment metadata. You should verify this is appropriate. Incorrect parent settings may lead to invalid CMIP submission.
  warnings.warn(


Model:,ACCESS-ESM1.6
Variable:,tas
CF Standard Name:,air_temperature
Units:,K
Dimensions:,"time: time, lat: lat, lon: lon"
Model Variable:,fld_s03i236
Processing:,DIRECT
Formula:,fld_s03i236


### Working with Ocean Variables

ACCESS-MOPPy also supports ocean variables from the ACCESS-ESM1.6 ocean component. Ocean data typically requires different file selection strategies, as ocean variables are stored in different file patterns and locations compared to atmospheric data.

Here's how to work with ocean salinity (so) from the ocean monthly table (Omon):

In [ ]:
# Import the ocean file discovery utility function from access_moppy
from access_moppy.utilities import get_monthly_ocean_files

# Find ocean salinity files using the utility function with explicit paths
ocean_files = get_monthly_ocean_files(
    "Omon.so", 
    root_folder=ROOT_FOLDER,
    target_folders=OCEAN_FOLDERS
)
print(f"Ocean files: {ocean_files[:2]}...")  # Show first 2 files

In [ ]:
# Create a CMORiser for ocean salinity
ocean_cmoriser = ACCESS_ESM_CMORiser(
    input_data=ocean_files,
    compound_name="Omon.so",  # Ocean salinity
    experiment_id="piControl-spinup",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,
)

# Display the ocean variable mapping
ocean_cmoriser.variable_mapping

#### Ocean Grid Variables (Fixed Fields)

Ocean grid variables like cell areas and depths are stored in different files and use the `Ofx` (Ocean fixed) table. These are time-independent variables that describe the ocean grid geometry:

In [ ]:
# Example: Ocean cell area (areacello)
# Use the predefined folder paths for consistency
grid_files = [
    ROOT_FOLDER + OCEAN_GRID_FOLDERS + "ocean-2d-area_t.nc",
    ROOT_FOLDER + OCEAN_GRID_FOLDERS + "ocean-2d-ht.nc",  # Needed for depth info
]

# Create CMORiser for ocean cell area
areacello_cmoriser = ACCESS_ESM_CMORiser(
    input_data=grid_files,
    compound_name="Ofx.areacello",  # Ocean cell area
    experiment_id="piControl-spinup",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,
)

# Display the mapping for ocean cell area
areacello_cmoriser.variable_mapping

#### Key Differences Between Model Components

Notice the differences in variable handling across ACCESS-ESM components:

1. **File Patterns**: 
   - Atmosphere: `aiihca.pa-*_mon.nc` (atmosphere component)
   - Ocean: `*-{variable}-1monthly-mean*.nc` (ocean component, specific variable naming)
   - Land: `aiihca.pe-*_mon.nc` (land component)

2. **CMIP Tables**:
   - `Amon`: Atmospheric monthly variables
   - `Omon`: Ocean monthly variables  
   - `Ofx`: Ocean fixed (time-independent) variables
   - `Lmon`: Land monthly variables (physical land surface)
   - `Emon`: Earth system monthly variables (biogeochemical/ecological)

3. **File Selection**: 
   - Atmospheric and land variables: Use component-specific file patterns
   - Ocean variables: Require knowledge of underlying model variable names through mappings

4. **Grid Information**: 
   - Atmospheric and land: Typically self-contained grid information
   - Ocean: Often requires additional files for proper coordinate information

### Working with Land Variables

ACCESS-MOPPy also supports land variables from the ACCESS-ESM1.6 land component. Land variables are organized into different CMIP tables depending on their scientific domain:

- **Lmon**: Standard land monthly variables (e.g., soil moisture, runoff, vegetation)
- **Emon**: Earth system monthly variables (e.g., carbon fluxes, fire, vegetation dynamics)

Land data in ACCESS models typically comes from the land component output files with their own naming patterns:

In [ ]:
# Example 1: Land Monthly Variable - Surface Runoff (mrro)
# Use the predefined folder paths for consistency
land_files = glob.glob(ROOT_FOLDER + LAND_FOLDERS + "aiihca.pe-*_mon.nc")
print(f"Found {len(land_files)} land files")

# Create CMORiser for land surface runoff
land_cmoriser = ACCESS_ESM_CMORiser(
    input_data=land_files,
    compound_name="Lmon.mrro",  # Surface runoff
    experiment_id="piControl-spinup",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,
)

# Display the land variable mapping
land_cmoriser.variable_mapping

In [ ]:
# Example 2: Earth System Monthly Variable - Gross Primary Productivity (gpp)
# Carbon cycle variables are also in land component files but use Emon table
emon_cmoriser = ACCESS_ESM_CMORiser(
    input_data=land_files,  # Can reuse the same land files
    compound_name="Emon.gpp",  # Gross Primary Productivity
    experiment_id="piControl-spinup",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,
)

# Display the Emon variable mapping
emon_cmoriser.variable_mapping

#### Land Variable Categories

Land variables in ACCESS-ESM are organized into different CMIP tables based on their scientific focus:

1. **Lmon (Land Monthly)**:
   - Standard land surface variables
   - Examples: `mrro` (runoff), `mrso` (soil moisture), `tsl` (soil temperature)
   - Physical land surface processes

2. **Emon (Earth System Monthly)**:
   - Biogeochemical and ecological variables  
   - Examples: `gpp` (gross primary productivity), `npp` (net primary productivity), `cVeg` (vegetation carbon)
   - Carbon cycle and ecosystem dynamics

3. **File Patterns**:
   - Land files: `aiihca.pe-*_mon.nc` (land component output)
   - Same files can contain variables for both Lmon and Emon tables
   - Variables are differentiated by the CMIP table specification, not the source files

### CMIP7 Support with Full Branded Names

ACCESS-MOPPy also supports the new CMIP7 compound name format, which uses full branded names instead of the table.variable format used in CMIP6. CMIP7 introduces a more descriptive naming convention that includes detailed information about the data processing and grid specifications.

The CMIP7 format follows the pattern: `realm.variable.operation.frequency.domain` (e.g., `atmos.tas.tavg-h2m-hxy-u.mon.glb`)

This provides more explicit information about:
- **Realm**: Model component (atmos, ocean, land, etc.)
- **Variable**: The physical quantity
- **Operation**: Temporal and spatial processing applied
- **Frequency**: Data output frequency
- **Domain**: Spatial domain specification

Let's see how to use CMIP7 compound names with ACCESS-MOPPy:

In [ ]:
# Example: CMIP7 compound name for atmospheric temperature
# Note the different source_id and the explicit cmip_version parameter
cmip7_cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,  # Can reuse the same atmospheric files
    compound_name="atmos.tas.tavg-h2m-hxy-u.mon.glb",  # CMIP7 full branded name
    experiment_id="piControl-spinup",
    source_id="pcmdi-test-1-0",  # CMIP7 test source identifier <-- This is a placeholder as ACCESS-ESM1-6 is yet to be registered in CMIP7 Vocabulary
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,
    cmip_version="CMIP7"  # <---- Explicit CMIP7 support (Default is still CMIP6)
)

# Display the CMIP7 variable mapping
cmip7_cmoriser.variable_mapping

#### CMIP6 vs CMIP7 Compound Name Comparison

The table below shows the key differences between CMIP6 and CMIP7 compound name formats:

| Aspect | CMIP6 Format | CMIP7 Format |
|--------|-------------|-------------|
| **Structure** | `table.variable` | `realm.variable.operation.frequency.domain` |
| **Example** | `Amon.tas` | `atmos.tas.tavg-h2m-hxy-u.mon.glb` |
| **Information** | Table and variable only | Detailed processing and grid info |
| **Length** | Compact | More descriptive |

**CMIP7 Compound Name Breakdown:**
- `atmos`: Atmospheric realm
- `tas`: Near-surface air temperature
- `tavg-h2m-hxy-u`: Time-averaged, 2-meter height, horizontal grid, unstructured
- `mon`: Monthly frequency
- `glb`: Global domain

ACCESS-MOPPy automatically handles the mapping between these formats and the underlying model variables.

## Running the CMORiser

To start the CMORisation process, simply call the `run()` method on your `cmoriser` instance as shown below. This step may take some time, especially if you are processing a large number of files.

We recommend using the [dask-labextension](https://github.com/dask/dask-labextension) with JupyterLab to monitor the progress of your computation. The extension provides a convenient dashboard to track task progress and resource usage directly within your notebook interface.


In [10]:
cmoriser.run()

🗓️  Monthly CMIP6 table detected (Amon.rsds) - using calendar-aware validation
🚀 Sampling 10 files from 12 total for efficient frequency detection
📂 Opening 8 files with xarray multi-file dataset...


/home/romain/PROJECTS/ACCESS-MOPPeR/ACCESS-MOPPy/src/access_moppy/utilities.py:622: UserWarning: Multi-file concatenation failed (conflicting values for variable 'surface_altitude' on objects to be combined. You can skip this check by specifying compat='override'.), falling back to individual file analysis
  warnings.warn(


📁 Analyzing 8 files individually...
🎯 Detected frequency from time bounds: 31 days 00:00:00
🎯 Detected frequency from time bounds: 31 days 00:00:00
🎯 Detected frequency from time bounds: 31 days 00:00:00
🎯 Detected frequency from time bounds: 31 days 00:00:00
🎯 Detected frequency from time bounds: 28 days 00:00:00
🎯 Detected frequency from time bounds: 31 days 00:00:00
🎯 Detected frequency from time bounds: 31 days 00:00:00
🎯 Detected frequency from time bounds: 30 days 00:00:00
📊 Detected frequency from individual files: 31 days 00:00:00
📅 Validated monthly data with calendar variations (detected: 31 days 00:00:00)
📅 Monthly data detected - no resampling required (calendar variations are natural)
✓ Validated compatible temporal frequency: 31 days 00:00:00
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for rsds...
✅ Final CMIP6 missing val

### In-memory processing with xarray and Dask

The CMORisation workflow processes data entirely in memory using `xarray` and Dask. This approach enables efficient parallel computation and flexible data manipulation, but requires that your system has enough memory to handle the size of your dataset. 

Once the CMORisation is complete, you can access the resulting dataset by calling the `to_dataset()` method on your `cmoriser` instance (see below). The returned object is a standard xarray dataset, which means you can slice, analyze, or further process the data using familiar xarray operations.

In [11]:
ds = cmoriser.to_dataset()

In [12]:
ds

<xarray.Dataset> Size: 1MB
Dimensions:    (lat: 145, bnds: 2, lon: 192, time: 12)
Coordinates:
  * lat        (lat) float64 1kB -90.0 -88.75 -87.5 -86.25 ... 87.5 88.75 90.0
  * lon        (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 354.4 356.2 358.1
  * time       (time) float64 96B 3.506e+05 3.507e+05 ... 3.51e+05 3.51e+05
Dimensions without coordinates: bnds
Data variables:
    lat_bnds   (lat, bnds) float64 2kB dask.array<chunksize=(145, 2), meta=np.ndarray>
    lon_bnds   (lon, bnds) float64 3kB dask.array<chunksize=(192, 2), meta=np.ndarray>
    time_bnds  (time, bnds) float64 192B dask.array<chunksize=(1, 2), meta=np.ndarray>
    rsds       (lon, lat, time) float32 1MB dask.array<chunksize=(192, 145, 1), meta=np.ndarray>
Attributes: (12/44)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    creation_date:          2026-02-19T00:46:54Z
    data_specs_version:     01.00.33
    experiment:             pre-industrial control (spin-up)
    experiment_id:          piControl-spinup
    ...                     ...
    branch_method:          standard
    external_variables:     areacella
    creator_name:           Romain Beucher
    creator_organisation:   ACCESS-NRI
    creator_email:          romain.beucher@anu.edu.au
    creator_url:            www.google.com

### Writing the output to a NetCDF file

To save your CMORised data to disk, use the `write()` method of the `cmoriser` instance. This will create a NetCDF file with all attributes set according to the CMIP Controlled Vocabulary, ensuring compliance with CMIP metadata standards.

After writing the file, we recommend validating it using [PrePARE](https://github.com/PCMDI/cmor/tree/master/PrePARE), a tool provided by PCMDI to check the conformity of CMIP files. PrePARE will help you identify any issues with metadata or file structure before publication or further analysis.

In [13]:
cmoriser.write()

Data size: 0.00 GB, Available memory: 1.17 GB
CMORised output written to rsds_Amon_ACCESS-ESM1-5_piControl-spinup_r1i1p1f1_gn_096101-096112.nc
📁 Optimized layout: metadata → data chunks
🗜️ HDF5 compression: shuffle + zlib(level 4) + fletcher32 for data variables
